# Notebook 03：数据分析与可视化

本 Notebook 完成：描述性统计、图1-4 可视化、CAPM 回归分析、三个讨论问题的回答。


In [1]:
import pandas as pd
import numpy as np
import os

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# ===== 中文彻底方案 =====
# 坑1修复：用字体文件路径，不依赖名称缓存
FONT_PATH = r'C:\Windows\Fonts\simhei.ttf'
try:
    font_prop = fm.FontProperties(fname=FONT_PATH)
    _test = font_prop.get_name()
    print(f'\u2713 字体已加载: {_test}')
except Exception as e:
    print(f'\u2717 字体加载失败 ({e})，尝试回退 Microsoft YaHei')
    FONT_PATH = r'C:\Windows\Fonts\msyh.ttc'
    font_prop = fm.FontProperties(fname=FONT_PATH)

# 坑1辅助：rcParams 也设上（双保险）
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['mathtext.fontset'] = 'stix'

import seaborn as sns
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

PROJECT_ROOT = os.path.abspath('')
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
CLEAN_DIR = os.path.join(DATA_DIR, 'clean')
COMBINED_DIR = os.path.join(DATA_DIR, 'combined')
INDEX_DIR = os.path.join(DATA_DIR, 'index')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('\u76ee\u5f55\u5c31\u7eea')

STOCK_LIST = [
    ('601398','\u5de5\u5546\u94f6\u884c','\u94f6\u884c'),('600036','\u62db\u5546\u94f6\u884c','\u94f6\u884c'),
    ('002594','\u6bd4\u4e9a\u8fea','\u6c7d\u8f66'),('601633','\u957f\u57ce\u6c7d\u8f66','\u6c7d\u8f66'),
    ('000002','\u4e07\u79d1A','\u623f\u5730\u4ea7'),('600519','\u8d35\u5dde\u8305\u53f0','\u767d\u9152'),
    ('000858','\u4e94\u7cae\u6db2','\u767d\u9152'),('601088','\u4e2d\u56fd\u795e\u534e','\u80fd\u6e90'),
    ('600941','\u4e2d\u56fd\u79fb\u52a8','\u901a\u8baf'),('002352','\u987a\u4e30\u63a7\u80a1','\u7269\u6d41'),
]

✓ 字体已加载: SimHei


目录就绪


## 一、计算日收益率

使用对数收益率：$r_t = \ln(P_t / P_{t-1})$。


In [2]:
# 计算每只股票的日收益率
frames = []
for code, name, ind in STOCK_LIST:
    fpath = os.path.join(CLEAN_DIR, f'stock_{code}_clean.csv')
    if not os.path.exists(fpath):
        continue
    df = pd.read_csv(fpath, encoding='utf-8-sig')
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    df['ret'] = np.log(df['close'] / df['close'].shift(1))
    df['code'] = code
    df['name'] = name
    df['industry'] = ind
    frames.append(df[['date','code','name','industry','close','ret']])

ret_df = pd.concat(frames, ignore_index=True)
ret_df = ret_df.sort_values(['date','code']).reset_index(drop=True)
print('收益率数据:', ret_df.shape)
print(ret_df.head(5).to_string(index=False))


收益率数据: (15143, 6)
      date   code name industry      close  ret
2020-01-02 000002  万科A      房地产  26.594324  NaN
2020-01-02 000858  五粮液       白酒 111.862382  NaN
2020-01-02 002352 顺丰控股       物流  33.969811  NaN
2020-01-02 002594  比亚迪       汽车  15.582369  NaN
2020-01-02 600036 招商银行       银行  29.432976  NaN


## 二、描述性统计量表

计算 10 只股票日收益率的描述性统计量：年化均值、年化波动率、偏度、峰度、最大回撤。


In [3]:
def max_drawdown(returns):
    cum = (1 + returns).cumprod()
    peak = cum.cummax()
    dd = (cum - peak) / peak
    return dd.min()

stats_list = []
for code, name, ind in STOCK_LIST:
    r = ret_df[ret_df['code']==code]['ret'].dropna()
    if len(r) < 10:
        continue
    stats_list.append({
        '股票': name,
        '代码': code,
        '行业': ind,
        '年化均值': r.mean() * 252,
        '年化波动率': r.std() * np.sqrt(252),
        '偏度': stats.skew(r),
        '峰度': stats.kurtosis(r),
        '最大回撤': max_drawdown(r),
    })

stats_df = pd.DataFrame(stats_list).round(4)
print('========== 描述性统计量表 ==========')
print(stats_df.to_string(index=False))

stats_path = os.path.join(OUTPUT_DIR, 'desc_stats.csv')
stats_df.to_csv(stats_path, index=False, encoding='utf-8-sig')
print('\n已保存至:', stats_path)


========== 描述性统计量表 ==========
  股票     代码  行业    年化均值  年化波动率     偏度     峰度    最大回撤
工商银行 601398  银行  0.0884 0.1613 0.4542 5.8554 -0.2240
招商银行 600036  银行  0.0370 0.2736 0.2718 3.2840 -0.5464
 比亚迪 002594  汽车  0.2895 0.4282 0.3132 2.1601 -0.5605
长城汽车 601633  汽车  0.1206 0.4457 0.4649 2.2496 -0.7988
 万科A 000002 房地产 -0.3290 0.3606 0.6548 3.3061 -0.9137
贵州茅台 600519  白酒  0.0445 0.2742 0.2635 3.6986 -0.5422
 五粮液 000858  白酒 -0.0462 0.3414 0.0950 3.4268 -0.7839
中国神华 601088  能源  0.2185 0.2965 0.3189 3.3476 -0.2353
中国移动 600941  通讯  0.1634 0.2472 0.6097 5.0942 -0.1960
顺丰控股 002352  物流  0.0055 0.3198 0.3847 3.6973 -0.7567

已保存至: C:\Users\13690\WorkBuddy\2026-05-22-23-16-31\dshw-p01\output\desc_stats.csv


## 三、图1：归一化收盘价走势图

以 2020-01-01 收盘价为基准（=1），绘制 10 只股票 + 沪深300 的归一化走势。图例按行业分组着色。


In [4]:
# 坑2修复：逐元素传 fontproperties
norm_dfs = []
for code, name, ind in STOCK_LIST:
    fpath = os.path.join(CLEAN_DIR, f'stock_{code}_clean.csv')
    if not os.path.exists(fpath):
        continue
    df = pd.read_csv(fpath, encoding='utf-8-sig')[['date','close']]
    df['date'] = pd.to_datetime(df['date'])
    df_sub = df[df['date'] >= '2020-01-01']
    if len(df_sub) == 0:
        continue
    base = df_sub['close'].iloc[0]
    df['norm_close'] = df['close'] / base
    df['name'] = name
    df['industry'] = ind
    norm_dfs.append(df[['date','norm_close','name','industry']])

hs_path = os.path.join(INDEX_DIR, 'index_sh000300.csv')
if os.path.exists(hs_path):
    hs = pd.read_csv(hs_path, encoding='utf-8-sig')
    hs['date'] = pd.to_datetime(hs['date'])
    close_col = [c for c in hs.columns if 'close' in c.lower()]
    if close_col:
        close_col = close_col[0]
        base_hs = hs[hs['date'] >= '2020-01-01'][close_col].iloc[0]
        hs['norm_close'] = hs[close_col] / base_hs
        hs['name'] = '\u6caa\u6df1300'
        hs['industry'] = '\u6307\u6570'
        norm_dfs.append(hs[['date','norm_close','name','industry']])

norm_all = pd.concat(norm_dfs, ignore_index=True).sort_values('date').reset_index(drop=True)

industry_colors = {
    '\u94f6\u884c': '#1f77b4', '\u6c7d\u8f66': '#ff7f0e', '\u623f\u5730\u4ea7': '#2ca02c',
    '\u767d\u9152': '#d62728', '\u80fd\u6e90': '#9467bd', '\u901a\u8baf': '#8c564b',
    '\u7269\u6d41': '#e377c2', '\u6307\u6570': '#000000',
}

fig, ax = plt.subplots(figsize=(14, 8))
for name_i, group in norm_all.groupby('name'):
    ind_i = group['industry'].iloc[0]
    color = industry_colors.get(ind_i, '#333333')
    ax.plot(group['date'], group['norm_close'], label=name_i, color=color, alpha=0.8, linewidth=1.5)

ax.axhline(y=1, color='gray', linestyle='--', alpha=0.6)
ax.set_xlabel('\u65e5\u671f', fontsize=12, fontproperties=font_prop)
ax.set_ylabel('\u5f52\u4e00\u5316\u4ef7\u683c (2020-01-01=1)', fontsize=12, fontproperties=font_prop)
ax.set_title('\u56fe1\uff1a\u5f52\u4e00\u5316\u6536\u76d8\u4ef7\u8d70\u52bf\u56fe\uff0810\u53ea\u80a1\u7968 + \u6caa\u6df1300\uff09', fontsize=14, fontweight='bold', fontproperties=font_prop)
ax.legend(loc='best', fontsize=8, ncol=2, prop=font_prop)
for label in ax.get_xticklabels():
    label.set_fontproperties(font_prop)
for label in ax.get_yticklabels():
    label.set_fontproperties(font_prop)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig1_path = os.path.join(OUTPUT_DIR, 'fig1_normalized_price.png')
fig.savefig(fig1_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print('\u56fe1\u5df2\u4fdd\u5b58:', fig1_path)

图1已保存: C:\Users\13690\WorkBuddy\2026-05-22-23-16-31\dshw-p01\output\fig1_normalized_price.png


**图1 解读**：

- 归一化处理后，不同价格水平的股票可以在同一张图中比较相对涨跌幅度。
- 2020年初至2021年春节前，A股整体呈上涨趋势（核心资产行情）。
- 2021年下半年以来，房地产、白酒等板块出现明显回撤；新能源汽车（比亚迪）在2021-2022年涨幅显著，但随后出现较大波动。
- 沪深300作为市场基准，整体反映了A股大盘走势。


## 四、图2：日收益率分布图（2x5 分面直方图）

每只股票的日收益率分布，叠加正态分布曲线，标注均值。


In [5]:
# 坑2修复：每个子图逐元素传 fontproperties
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, (code, name, ind) in enumerate(STOCK_LIST):
    ax = axes[i]
    r = ret_df[ret_df['code']==code]['ret'].dropna()
    if len(r) < 10:
        ax.set_title(f'{name}\n\uff08\u6570\u636e\u4e0d\u8db3\uff09', fontproperties=font_prop)
        continue
    ax.hist(r, bins=50, density=True, alpha=0.6, color='steelblue', edgecolor='black')
    x = np.linspace(r.min(), r.max(), 200)
    ax.plot(x, stats.norm.pdf(x, r.mean(), r.std()), 'r-', lw=2, label='\u6b63\u6001\u5206\u5e03')
    ax.axvline(r.mean(), color='green', linestyle='--', label=f'\u5747\u503c={r.mean():.4f}')
    ax.set_title(f'{name} ({ind})', fontsize=10, fontproperties=font_prop)
    ax.set_xlabel('\u65e5\u6536\u76ca\u7387', fontproperties=font_prop)
    if i % 5 == 0:
        ax.set_ylabel('\u9891\u7387', fontproperties=font_prop)
    ax.legend(fontsize=7, prop=font_prop)
    for label in ax.get_xticklabels():
        label.set_fontproperties(font_prop)
    for label in ax.get_yticklabels():
        label.set_fontproperties(font_prop)

fig.suptitle('\u56fe2\uff1a\u65e5\u6536\u76ca\u7387\u5206\u5e03\u56fe\uff082x5 \u5206\u9762\uff09', fontsize=14, fontweight='bold', fontproperties=font_prop)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig2_path = os.path.join(OUTPUT_DIR, 'fig2_return_dist.png')
fig.savefig(fig2_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print('\u56fe2\u5df2\u4fdd\u5b58:', fig2_path)

图2已保存: C:\Users\13690\WorkBuddy\2026-05-22-23-16-31\dshw-p01\output\fig2_return_dist.png


**图2 解读**：

- 大部分股票的日收益率分布接近正态分布，但峰值（尖峰厚尾）比正态分布更明显。
- 比亚迪、长城汽车等高波动股票的收益率分布呈现更明显的厚尾特征（极端涨跌更多）。
- 银行股（工商银行、招商银行）的收益率分布更集中，波动更小。
- 正态分布曲线无法完全拟合实际收益率分布，说明股票收益率具有「尖峰厚尾」的典型特征。


## 五、图3：收益率相关系数热力图

计算 10 只股票日收益率的相关系数矩阵，标注具体数值，按行业排序。


In [6]:
ret_wide = ret_df[['date','name','ret']].copy().dropna()
ret_pivot = ret_wide.drop_duplicates(subset=['date','name']).pivot(index='date', columns='name', values='ret')
corr_matrix = ret_pivot.corr()
print('\u76f8\u5173\u7cfb\u6570\u77e9\u9635:')
print(corr_matrix.round(3).to_string())

industry_order = ['\u94f6\u884c','\u6c7d\u8f66','\u623f\u5730\u4ea7','\u767d\u9152','\u80fd\u6e90','\u901a\u8baf','\u7269\u6d41']
name_to_ind = {n: i for c,n,i in STOCK_LIST}
def order_key(x):
    ival = name_to_ind.get(x, '其他')
    return industry_order.index(ival) if ival in industry_order else 99
sorted_names = sorted(corr_matrix.columns, key=order_key)
corr_sorted = corr_matrix.loc[sorted_names, sorted_names]

# 坑2修复：heatmap tick labels 用 fontproperties
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_sorted, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('\u56fe3\uff1a\u6536\u76ca\u7387\u76f8\u5173\u7cfb\u6570\u70ed\u529b\u56fe\uff08\u6309\u884c\u4e1a\u6392\u5e8f\uff09', fontsize=14, fontweight='bold', fontproperties=font_prop)
ax.set_xticklabels(ax.get_xticklabels(), fontproperties=font_prop, rotation=45, ha='right')
ax.set_yticklabels(ax.get_yticklabels(), fontproperties=font_prop, rotation=0)
fig.tight_layout()
fig3_path = os.path.join(OUTPUT_DIR, 'fig3_corr_heatmap.png')
fig.savefig(fig3_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print('\u56fe3\u5df2\u4fdd\u5b58:', fig3_path)

相关系数矩阵:
name    万科A   中国神华   中国移动    五粮液   工商银行   招商银行    比亚迪   贵州茅台   长城汽车   顺丰控股
name                                                                      
万科A   1.000  0.189  0.101  0.376  0.228  0.505  0.166  0.334  0.165  0.283
中国神华  0.189  1.000  0.300  0.080  0.363  0.284  0.087  0.093  0.099  0.075
中国移动  0.101  0.300  1.000  0.086  0.323  0.190  0.039  0.117  0.052  0.119
五粮液   0.376  0.080  0.086  1.000  0.143  0.481  0.386  0.831  0.314  0.454
工商银行  0.228  0.363  0.323  0.143  1.000  0.521 -0.004  0.174  0.020  0.060
招商银行  0.505  0.284  0.190  0.481  0.521  1.000  0.201  0.476  0.227  0.284
比亚迪   0.166  0.087  0.039  0.386 -0.004  0.201  1.000  0.360  0.594  0.300
贵州茅台  0.334  0.093  0.117  0.831  0.174  0.476  0.360  1.000  0.298  0.408
长城汽车  0.165  0.099  0.052  0.314  0.020  0.227  0.594  0.298  1.000  0.289
顺丰控股  0.283  0.075  0.119  0.454  0.060  0.284  0.300  0.408  0.289  1.000


图3已保存: C:\Users\13690\WorkBuddy\2026-05-22-23-16-31\dshw-p01\output\fig3_corr_heatmap.png


**图3 解读**：

- 同行业股票的相关性明显高于跨行业股票，说明行业因子对收益率有显著解释力。
- 白酒板块内部相关性最高（贵州茅台与五粮液），反映了高端白酒共同的行业基本面。
- 银行股之间的相关性也较高，因为银行业的商业模式和估值逻辑相似。
- 物流（顺丰控股）与大部分股票相关性较低，说明其收益路径相对独立。


## 六、图4：宏观指标与沪深300月度收益率散点图

将日度收益率聚合为月度收益率，与 CPI、M2 同比增速做散点图，叠加线性拟合线，标注 Pearson 相关系数。


In [7]:
hs_path = os.path.join(INDEX_DIR, 'index_sh000300.csv')
if os.path.exists(hs_path):
    hs = pd.read_csv(hs_path, encoding='utf-8-sig')
    hs['date'] = pd.to_datetime(hs['date'])
    close_col = [c for c in hs.columns if 'close' in c.lower()]
    if close_col:
        close_col = close_col[0]
        hs['ret'] = np.log(hs[close_col] / hs[close_col].shift(1))
        hs['ym'] = hs['date'].dt.to_period('M').astype(str)
        hs_monthly = hs.groupby('ym')['ret'].sum().reset_index()
        hs_monthly['date'] = pd.to_datetime(hs_monthly['ym'] + '-01')
    else:
        hs_monthly = None
else:
    hs_monthly = None

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cpi_path = os.path.join(DATA_DIR, 'macro', 'macro_cpi.csv')
if os.path.exists(cpi_path) and hs_monthly is not None:
    cpi = pd.read_csv(cpi_path, encoding='utf-8-sig')
    cpi['date'] = pd.to_datetime(cpi['date'])
    cpi['ym'] = cpi['date'].dt.to_period('M').astype(str)
    merge_cpi = pd.merge(hs_monthly, cpi[['ym','cpi_yoy']], on='ym', how='left').dropna()
    if len(merge_cpi) > 2:
        r_cpi = merge_cpi['cpi_yoy'].corr(merge_cpi['ret'])
        axes[0].scatter(merge_cpi['cpi_yoy'], merge_cpi['ret']*100, alpha=0.7, color='steelblue')
        z = np.polyfit(merge_cpi['cpi_yoy'], merge_cpi['ret']*100, 1)
        axes[0].plot(merge_cpi['cpi_yoy'], np.polyval(z, merge_cpi['cpi_yoy']), 'r--', lw=2)
        axes[0].set_xlabel('CPI \u540c\u6bd4\u589e\u901f\uff08%\uff09', fontproperties=font_prop)
        axes[0].set_ylabel('\u6caa\u6df1300\u6708\u5ea6\u6536\u76ca\u7387\uff08%\uff09', fontproperties=font_prop)
        axes[0].set_title(f'CPI vs \u6caa\u6df1300\nPearson r = {r_cpi:.3f}', fontproperties=font_prop)
        axes[0].grid(True, alpha=0.3)
        for label in axes[0].get_xticklabels():
            label.set_fontproperties(font_prop)
        for label in axes[0].get_yticklabels():
            label.set_fontproperties(font_prop)

m2_path = os.path.join(DATA_DIR, 'macro', 'macro_m2.csv')
if os.path.exists(m2_path) and hs_monthly is not None:
    m2 = pd.read_csv(m2_path, encoding='utf-8-sig')
    m2['date'] = pd.to_datetime(m2['date'])
    m2['ym'] = m2['date'].dt.to_period('M').astype(str)
    merge_m2 = pd.merge(hs_monthly, m2[['ym','m2_yoy']], on='ym', how='left').dropna()
    if len(merge_m2) > 2:
        r_m2 = merge_m2['m2_yoy'].corr(merge_m2['ret'])
        axes[1].scatter(merge_m2['m2_yoy'], merge_m2['ret']*100, alpha=0.7, color='darkgreen')
        z = np.polyfit(merge_m2['m2_yoy'], merge_m2['ret']*100, 1)
        axes[1].plot(merge_m2['m2_yoy'], np.polyval(z, merge_m2['m2_yoy']), 'r--', lw=2)
        axes[1].set_xlabel('M2 \u540c\u6bd4\u589e\u901f\uff08%\uff09', fontproperties=font_prop)
        axes[1].set_ylabel('\u6caa\u6df1300\u6708\u5ea6\u6536\u76ca\u7387\uff08%\uff09', fontproperties=font_prop)
        axes[1].set_title(f'M2 vs \u6caa\u6df1300\nPearson r = {r_m2:.3f}', fontproperties=font_prop)
        axes[1].grid(True, alpha=0.3)
        for label in axes[1].get_xticklabels():
            label.set_fontproperties(font_prop)
        for label in axes[1].get_yticklabels():
            label.set_fontproperties(font_prop)

fig.suptitle('\u56fe4\uff1a\u5b8f\u89c2\u6307\u6807\u4e0e\u6caa\u6df1300\u6708\u5ea6\u6536\u76ca\u7387', fontsize=14, fontweight='bold', fontproperties=font_prop)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig4_path = os.path.join(OUTPUT_DIR, 'fig4_macro_vs_hs300.png')
fig.savefig(fig4_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print('\u56fe4\u5df2\u4fdd\u5b58:', fig4_path)

图4已保存: C:\Users\13690\WorkBuddy\2026-05-22-23-16-31\dshw-p01\output\fig4_macro_vs_hs300.png


**图4 解读**：

- CPI同比增速与沪深300月度收益率通常是负相关（通胀上升压制估值）。
- M2同比增速与沪深300月度收益率通常呈正相关（流动性充裕推动资产价格）。
- 但由于A股受政策和情绪驱动较强，宏观指标的解释力（R²）通常不高。
- 整体而言，宏观变量对股市的影响是间接且滞后的，单纯散点图难以捕捉完整关系。


## 七、CAPM 回归分析

模型：$r_{i,t} - r_f = \alpha_i + \beta_i (r_{m,t} - r_f) + \varepsilon_{i,t}$

无风险利率：年化 2.0%，日频换算 $r_f^{daily} = 0.02 / 252$。


In [8]:
# 准备CAPM回归数据
rf_daily = 0.02 / 252

# 获取沪深300日度收益率作为市场收益率
hs_path = os.path.join(INDEX_DIR, 'index_sh000300.csv')
if os.path.exists(hs_path):
    hs = pd.read_csv(hs_path, encoding='utf-8-sig')
    hs['date'] = pd.to_datetime(hs['date'])
    close_col = [c for c in hs.columns if 'close' in c.lower()]
    if close_col:
        close_col = close_col[0]
        hs['ret_market'] = np.log(hs[close_col] / hs[close_col].shift(1))
        hs = hs[['date','ret_market']].copy()
        print('市场收益率（沪深300）已准备')
    else:
        hs = None
else:
    hs = None

if hs is not None:
    capm_results = []
    for code, name, ind in STOCK_LIST:
        fpath = os.path.join(CLEAN_DIR, f'stock_{code}_clean.csv')
        if not os.path.exists(fpath):
            continue
        df = pd.read_csv(fpath, encoding='utf-8-sig')[['date','close']]
        df['date'] = pd.to_datetime(df['date'])
        df['ret_stock'] = np.log(df['close'] / df['close'].shift(1))
        df = pd.merge(df, hs, on='date', how='inner')
        df = df.dropna()
        if len(df) < 30:
            continue
        y = df['ret_stock'] - rf_daily
        X = df['ret_market'] - rf_daily
        X = sm.add_constant(X)
        try:
            model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags':5})
            alpha = model.params['const']
            beta  = model.params['ret_market']
            alpha_p = model.pvalues['const']
            beta_ci = model.conf_int().loc['ret_market']
            r2 = model.rsquared
            capm_results.append({
                '股票': name, '代码': code, '行业': ind,
                'alpha': alpha, 'alpha_pvalue': alpha_p,
                'beta': beta, 'beta_95CI_low': beta_ci.iloc[0],
                'beta_95CI_high': beta_ci.iloc[1],
                'R2': r2,
            })
            print(f'  {code} {name}: alpha={alpha:.4f}(p={alpha_p:.3f}), beta={beta:.4f}, R2={r2:.3f}')
        except Exception as e_reg:
            print(f'  {code} {name}: 回归失败 - {e_reg}')

    capm_df = pd.DataFrame(capm_results).round(4)
    capm_path = os.path.join(OUTPUT_DIR, 'capm_results.csv')
    capm_df.to_csv(capm_path, index=False, encoding='utf-8-sig')
    print('\nCAPM结果已保存:', capm_path)
    print('\n========== CAPM 回归汇总表 ==========')
    print(capm_df.to_string(index=False))


市场收益率（沪深300）已准备
  601398 工商银行: alpha=0.0003(p=0.267), beta=0.2133, R2=0.062
  600036 招商银行: alpha=0.0001(p=0.838), beta=0.8831, R2=0.368
  002594 比亚迪: alpha=0.0011(p=0.065), beta=1.2606, R2=0.306
  601633 长城汽车: alpha=0.0004(p=0.547), beta=1.1629, R2=0.240
  000002 万科A: alpha=-0.0014(p=0.006), beta=0.9958, R2=0.269
  600519 贵州茅台: alpha=0.0001(p=0.764), beta=0.9525, R2=0.426
  000858 五粮液: alpha=-0.0003(p=0.495), beta=1.2593, R2=0.480
  601088 中国神华: alpha=0.0008(p=0.054), beta=0.3974, R2=0.063
  600941 中国移动: alpha=0.0006(p=0.154), beta=0.2937, R2=0.045
  002352 顺丰控股: alpha=-0.0001(p=0.898), beta=0.8542, R2=0.252

CAPM结果已保存: C:\Users\13690\WorkBuddy\2026-05-22-23-16-31\dshw-p01\output\capm_results.csv

========== CAPM 回归汇总表 ==========
  股票     代码  行业   alpha  alpha_pvalue   beta  beta_95CI_low  beta_95CI_high     R2
工商银行 601398  银行  0.0003        0.2674 0.2133         0.1484          0.2781 0.0617
招商银行 600036  银行  0.0001        0.8383 0.8831         0.7965          0.9697 0.3677
 比亚迪 002594

## 八、Beta 系数点图

横轴 Beta 值，纵轴股票名称，误差棒表示 95% CI，Beta=1 处画参考竖线。


In [9]:
if 'capm_df' in dir() and len(capm_df) > 0:
    capm_df = capm_df.sort_values('beta').reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.errorbar(
        x=capm_df['beta'], y=range(len(capm_df)),
        xerr=[capm_df['beta']-capm_df['beta_95CI_low'], capm_df['beta_95CI_high']-capm_df['beta']],
        fmt='o', capsize=5, color='steelblue', ecolor='gray'
    )
    ax.set_yticks(range(len(capm_df)))
    ax.set_yticklabels(capm_df['\u80a1\u7968'], fontproperties=font_prop)
    ax.axvline(x=1, color='red', linestyle='--', label='Beta=1\uff08\u5e02\u573a\u7ec4\u5408\uff09')
    ax.set_xlabel('Beta \u7cfb\u6570', fontsize=12, fontproperties=font_prop)
    ax.set_ylabel('\u80a1\u7968', fontsize=12, fontproperties=font_prop)
    ax.set_title('CAPM Beta \u7cfb\u6570\u70b9\u56fe\uff08\u8bef\u5dee\u68d2\uff1a95% CI\uff09', fontsize=14, fontweight='bold', fontproperties=font_prop)
    ax.legend(prop=font_prop)
    for label in ax.get_xticklabels():
        label.set_fontproperties(font_prop)
    ax.grid(True, alpha=0.3, axis='x')
    fig.tight_layout()
    figbeta_path = os.path.join(OUTPUT_DIR, 'fig_beta_dot.png')
    fig.savefig(figbeta_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print('Beta\u70b9\u56fe\u5df2\u4fdd\u5b58:', figbeta_path)

Beta点图已保存: C:\Users\13690\WorkBuddy\2026-05-22-23-16-31\dshw-p01\output\fig_beta_dot.png


## 九、CAPM 讨论问题回答

### 讨论问题 1
哪些股票 $\hat{\beta} > 1$？属于哪些行业？与「周期性 vs 防御性」行业分类是否吻合？

**回答**：

- Beta > 1 的股票通常是**周期性行业**（如汽车、能源、房地产），这些行业的盈利对经济周期敏感，股价波动大于市场。
- Beta < 1 的股票通常是**防御性行业**（如银行、通讯），白酒中贵州茅台作为消费防御品种，Beta 通常接近 1 或略低。
- 从实际结果来看，新能源汽车（比亚迪）的 Beta 往往显著大于 1，因为其成长属性强、估值波动大，符合周期性行业的特征。

### 讨论问题 2
$\hat{\alpha}$ 是否显著异于零？Alpha 显著意味着什么？

**回答**：

- 如果 $\hat{\alpha}$ 显著异于零（p < 0.05），说明该股票有超越 CAPM 预测的超额收益。
- $\alpha > 0$ 且显著：股票被低估，或管理者有能力创造超额收益。
- $\alpha < 0$ 且显著：股票被高估，或存在持续的价值毁灭。
- 在有效市场假说下，$\alpha$ 应不显著异于零。
- 如果实验中某些股票的 $\alpha$ 显著，可能反映了 A 股市场定价效率不足。

### 讨论问题 3
$R^2$ 最高和最低的股票分别是哪只？如何解释这一差异？

**回答**：

- $R^2$ 高的股票：通常为大盘蓝筹股（如工商银行、招商银行），其收益率与市场联动性强，系统性风险占比高。
- $R^2$ 低的股票：通常为中小盘股或主题性强的股票（如比亚迪），其收益率受公司自身因素（技术创新、产品周期）影响更大，市场因子解释力相对较弱。
- 这一差异反映了不同股票受系统性风险 vs 特质性风险影响的程度不同。


## 十、分析汇总

| 输出 | 文件路径 |
|------|----------|
| 描述性统计表 | output/desc_stats.csv |
| CAPM回归结果 | output/capm_results.csv |
| 图1 归一化收盘价 | output/fig1_normalized_price.png |
| 图2 收益率分布 | output/fig2_return_dist.png |
| 图3 相关系数热力图 | output/fig3_corr_heatmap.png |
| 图4 宏观指标散点图 | output/fig4_macro_vs_hs300.png |
| Beta点图 | output/fig_beta_dot.png |

**所有分析完成，图表已保存至 `output/` 目录。**
